# GraphVAE: generation of small molecular graphs
## Reproducing Simonovsky & Komodakis (2018)

This notebook explains and reproduces the central model from **GraphVAE: Towards Generation of Small Graphs Using Variational Autoencoders**.

- [Paper on arXiv](https://arxiv.org/abs/1802.03480)
- [Authors' implementation](https://github.com/martin-simonovsky/graphvae)

The reproduction uses QM9 heavy-atom graphs (at most nine atoms), a graph message-passing encoder, a one-shot probabilistic decoder, and permutation-aware reconstruction. It is configured for a CUDA cluster with AMP, large batches, cached tensors, early stopping, and reusable checkpoints. Processed inputs are stored in `data/GraphVAE/`; weights are stored in `models/`.

## 1. From a VAE to a graph VAE

A variational autoencoder learns an encoder $q_\phi(z\mid G)$ and decoder $p_\theta(G\mid z)$. The evidence lower bound is

$$\mathcal L = \mathbb E_{q_\phi(z|G)}[-\log p_\theta(G|z)] + \beta D_{KL}(q_\phi(z|G)\|\mathcal N(0,I)).$$

GraphVAE encodes an input graph using graph convolutions. Its decoder predicts an entire probabilistic graph in one shot: node existence/type distributions and edge existence/type distributions for a fixed maximum number of vertices. Samples from the Gaussian prior can therefore become new graphs without autoregressive decoding.

The awkward part is that vertex indices have no meaning. A target graph and an isomorphic permutation must have the same loss. The paper approximately matches decoded and target vertices before reconstruction. This implementation uses **differentiable Sinkhorn matching**, a GPU-native relaxation to a permutation matrix, instead of the paper's max-pooling graph-matching routine.

## 2. Molecular representation

Following the QM9 experiment, hydrogens are omitted and every molecule is padded to nine heavy atoms. Node classes are `PAD, C, N, O, F`; edge classes are `NONE, SINGLE, DOUBLE, TRIPLE, AROMATIC`. The decoder predicts only the upper triangle and mirrors it, guaranteeing an undirected graph.

This factorized decoder does not enforce chemical valence or connectivity. That is an important limitation of GraphVAE: syntactically probable node/edge decisions need not form a valid molecule.

In [ ]:
import copy
import math
import os
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torch_geometric.datasets import QM9

SEED = 23
random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = device.type == 'cuda'
if USE_AMP:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision('high')
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'GraphVAE' else Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'; MODELS_DIR = PROJECT_ROOT / 'models'
GRAPH_DATA_DIR = DATA_DIR / 'GraphVAE'
GRAPH_DATA_DIR.mkdir(parents=True, exist_ok=True); MODELS_DIR.mkdir(parents=True, exist_ok=True)
print(f'PyTorch {torch.__version__} | device={device}')
if USE_AMP:
    p = torch.cuda.get_device_properties(0)
    print(f'{p.name} | {p.total_memory/2**30:.1f} GiB | AMP/TF32 enabled')

## 3. Cache QM9 as fixed graph tensors

The first run converts PyG QM9 graphs and writes `data/GraphVAE/qm9_heavy_graphs.pt`. Later runs load that tensor cache directly, avoiding repeated Python graph preprocessing. PyG's downloaded QM9 files remain in `data/QM9/`.

In [ ]:
MAX_NODES = 9
ATOM_NUMBERS = [0, 6, 7, 8, 9]
ATOM_NAMES = ['PAD', 'C', 'N', 'O', 'F']
BOND_NAMES = ['NONE', 'SINGLE', 'DOUBLE', 'TRIPLE', 'AROMATIC']
N_NODE_TYPES, N_EDGE_TYPES = len(ATOM_NAMES), len(BOND_NAMES)
CACHE_PATH = GRAPH_DATA_DIR / 'qm9_heavy_graphs.pt'

def build_cache():
    qm9 = QM9(root=DATA_DIR / 'QM9')
    nodes = torch.zeros((len(qm9), MAX_NODES), dtype=torch.long)
    edges = torch.zeros((len(qm9), MAX_NODES, MAX_NODES), dtype=torch.long)
    atom_to_class = {z: i for i, z in enumerate(ATOM_NUMBERS)}
    for index, graph in enumerate(qm9):
        heavy_old = torch.where(graph.z > 1)[0]
        if len(heavy_old) > MAX_NODES:
            raise ValueError(f'Molecule {index} has {len(heavy_old)} heavy atoms')
        old_to_new = torch.full((graph.num_nodes,), -1, dtype=torch.long)
        old_to_new[heavy_old] = torch.arange(len(heavy_old))
        nodes[index, :len(heavy_old)] = torch.tensor([atom_to_class[int(z)] for z in graph.z[heavy_old]])
        src, dst = graph.edge_index
        keep = (old_to_new[src] >= 0) & (old_to_new[dst] >= 0)
        new_src, new_dst = old_to_new[src[keep]], old_to_new[dst[keep]]
        bond_class = graph.edge_attr[keep, :4].argmax(dim=-1) + 1
        edges[index, new_src, new_dst] = bond_class
        if (index + 1) % 25_000 == 0: print(f'Converted {index+1:,}/{len(qm9):,}')
    payload = {'node_types': nodes, 'edge_types': edges, 'atom_numbers': ATOM_NUMBERS,
               'max_nodes': MAX_NODES, 'version': 1}
    torch.save(payload, CACHE_PATH)
    return payload

cache = torch.load(CACHE_PATH, weights_only=True) if CACHE_PATH.exists() else build_cache()
if cache['max_nodes'] != MAX_NODES or cache['atom_numbers'] != ATOM_NUMBERS:
    raise ValueError('Incompatible cache; delete it and rerun this cell.')
node_types, edge_types = cache['node_types'], cache['edge_types']
print(f'Cached graphs: {len(node_types):,}; nodes={tuple(node_types.shape)}; cache={CACHE_PATH}')

In [ ]:
N_TRAIN, N_VAL = 110_000, 10_000
g = torch.Generator().manual_seed(SEED)
permutation = torch.randperm(len(node_types), generator=g)
train_idx = permutation[:N_TRAIN]
val_idx = permutation[N_TRAIN:N_TRAIN+N_VAL]
test_idx = permutation[N_TRAIN+N_VAL:]
BATCH_SIZE = 512 if USE_AMP else 64
NUM_WORKERS = min(8, os.cpu_count() or 1) if USE_AMP else 0
kwargs = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=USE_AMP,
              persistent_workers=NUM_WORKERS > 0)
def loader(indices, shuffle=False):
    return DataLoader(TensorDataset(node_types[indices], edge_types[indices]), shuffle=shuffle, **kwargs)
train_loader, val_loader, test_loader = loader(train_idx, True), loader(val_idx), loader(test_idx)
print(f'Split={len(train_idx):,}/{len(val_idx):,}/{len(test_idx):,}; batch={BATCH_SIZE}; workers={NUM_WORKERS}')

## 4. Encoder and one-shot decoder

The encoder performs bond-type-specific message passing and maps a pooled graph representation to $\mu(G)$ and $\log\sigma^2(G)$. Reparameterization gives $z=\mu+\sigma\odot\epsilon$. The MLP decoder emits all node and upper-triangular edge logits simultaneously.

In [ ]:
class GraphEncoder(nn.Module):
    def __init__(self, hidden=192, latent=64, layers=3):
        super().__init__()
        self.embedding = nn.Embedding(N_NODE_TYPES, hidden, padding_idx=0)
        self.transforms = nn.ModuleList([nn.ModuleList([nn.Linear(hidden, hidden, bias=False)
                                              for _ in range(N_EDGE_TYPES-1)]) for _ in range(layers)])
        self.updates = nn.ModuleList([nn.GRUCell(hidden, hidden) for _ in range(layers)])
        self.mu = nn.Linear(hidden, latent); self.logvar = nn.Linear(hidden, latent)
    def forward(self, nodes, edges):
        mask = nodes.ne(0); h = self.embedding(nodes)
        for transforms, update in zip(self.transforms, self.updates):
            message = torch.zeros_like(h)
            for bond, transform in enumerate(transforms, start=1):
                message = message + torch.bmm((edges == bond).to(h.dtype), transform(h))
            h = update(message.flatten(0, 1), h.flatten(0, 1)).view_as(h)
            h = h * mask.unsqueeze(-1)
        pooled = h.sum(1) / mask.sum(1, keepdim=True).sqrt().clamp_min(1)
        return self.mu(pooled), self.logvar(pooled).clamp(-10, 10)

class GraphDecoder(nn.Module):
    def __init__(self, latent=64, hidden=512):
        super().__init__()
        upper = torch.triu_indices(MAX_NODES, MAX_NODES, offset=1)
        self.register_buffer('row', upper[0]); self.register_buffer('col', upper[1])
        output_size = MAX_NODES*N_NODE_TYPES + upper.size(1)*N_EDGE_TYPES
        self.net = nn.Sequential(nn.Linear(latent, hidden), nn.ReLU(), nn.Linear(hidden, hidden),
                                 nn.ReLU(), nn.Linear(hidden, output_size))
    def forward(self, z):
        raw = self.net(z); split = MAX_NODES*N_NODE_TYPES
        node_logits = raw[:, :split].view(-1, MAX_NODES, N_NODE_TYPES)
        upper = raw[:, split:].view(-1, len(self.row), N_EDGE_TYPES)
        edge_logits = raw.new_zeros((len(z), MAX_NODES, MAX_NODES, N_EDGE_TYPES))
        edge_logits[:, self.row, self.col] = upper; edge_logits[:, self.col, self.row] = upper
        edge_logits[:, torch.arange(MAX_NODES), torch.arange(MAX_NODES), 0] = 20
        return node_logits, edge_logits

class GraphVAE(nn.Module):
    def __init__(self, latent=64):
        super().__init__(); self.latent = latent
        self.encoder = GraphEncoder(latent=latent); self.decoder = GraphDecoder(latent=latent)
    def forward(self, nodes, edges):
        mu, logvar = self.encoder(nodes, edges)
        z = mu + torch.exp(0.5*logvar) * torch.randn_like(mu) if self.training else mu
        return (*self.decoder(z), mu, logvar)

model = GraphVAE().to(device)
sample_nodes, sample_edges = next(iter(train_loader))
with torch.no_grad():
    out = model(sample_nodes[:4].to(device), sample_edges[:4].to(device))
print(f'Parameters={sum(p.numel() for p in model.parameters()):,}; node logits={out[0].shape}; edge logits={out[1].shape}')

## 5. Permutation-aware reconstruction

A Sinkhorn matrix $P$ softly assigns predicted slots to target slots using node-type and degree agreement. The aligned targets are $PX$ and $PAP^T$. Repeated row/column log normalization makes $P$ approximately doubly stochastic. Temperature is annealed toward a sharper assignment. Node and edge cross-entropies use lower weights for padding/no-edge classes to counter severe class imbalance.

In [ ]:
UPPER = torch.triu_indices(MAX_NODES, MAX_NODES, offset=1)
NODE_WEIGHTS = torch.tensor([0.25, 1, 1, 1, 1], device=device)
EDGE_WEIGHTS = torch.tensor([0.15, 1, 1, 1, 1], device=device)

def sinkhorn_match(node_logits, edge_logits, target_nodes, target_edges, temperature=0.25, steps=20):
    node_prob = node_logits.float().softmax(-1)
    edge_prob = edge_logits.float().softmax(-1)
    target_node_oh = F.one_hot(target_nodes, N_NODE_TYPES).float()
    target_edge_oh = F.one_hot(target_edges, N_EDGE_TYPES).float()
    predicted_degree = (1-edge_prob[..., 0]).sum(-1)
    target_degree = target_edges.ne(0).sum(-1).float()
    score = torch.einsum('bik,bjk->bij', node_prob, target_node_oh)
    score = score - 0.08*(predicted_degree[:, :, None]-target_degree[:, None, :]).abs()
    log_p = score / temperature
    for _ in range(steps):
        log_p = log_p - torch.logsumexp(log_p, dim=2, keepdim=True)
        log_p = log_p - torch.logsumexp(log_p, dim=1, keepdim=True)
    p = log_p.exp()
    aligned_nodes = torch.einsum('bij,bjk->bik', p, target_node_oh)
    aligned_edges = torch.einsum('bij,bjkc,blk->bilc', p, target_edge_oh, p)
    return aligned_nodes, aligned_edges, p

def graphvae_loss(node_logits, edge_logits, nodes, edges, mu, logvar, beta, temperature):
    aligned_n, aligned_e, _ = sinkhorn_match(node_logits, edge_logits, nodes, edges, temperature)
    node_logp = node_logits.float().log_softmax(-1)
    node_loss = -(aligned_n*NODE_WEIGHTS*node_logp).sum(-1).mean()
    r, c = UPPER.to(nodes.device)
    edge_logp = edge_logits[:, r, c].float().log_softmax(-1)
    edge_loss = -(aligned_e[:, r, c]*EDGE_WEIGHTS*edge_logp).sum(-1).mean()
    kl = -0.5*(1 + logvar.float() - mu.float().square() - logvar.float().exp()).sum(-1).mean()
    total = node_loss + edge_loss + beta*kl
    return total, {'node': node_loss.item(), 'edge': edge_loss.item(), 'kl': kl.item()}

In [ ]:
def run_epoch(loader, optimizer=None, epoch=1):
    training = optimizer is not None; model.train(training)
    totals = {'loss': 0., 'node': 0., 'edge': 0., 'kl': 0.}; count = 0
    beta = min(1.0, epoch/20); temperature = max(0.12, 0.5*(0.97**epoch))
    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for nodes, edges in loader:
            nodes = nodes.to(device, non_blocking=True); edges = edges.to(device, non_blocking=True)
            if training: optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
                node_logits, edge_logits, mu, logvar = model(nodes, edges)
                loss, parts = graphvae_loss(node_logits, edge_logits, nodes, edges, mu, logvar, beta, temperature)
            if training:
                scaler.scale(loss).backward(); scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                scaler.step(optimizer); scaler.update()
            batch = len(nodes); count += batch; totals['loss'] += loss.item()*batch
            for key in parts: totals[key] += parts[key]*batch
    return {key: value/count for key, value in totals.items()}

@torch.no_grad()
def reconstruction_accuracy(loader):
    model.eval(); correct_n = correct_e = total_n = total_e = 0
    for nodes, edges in loader:
        nodes, edges = nodes.to(device), edges.to(device)
        nl, el, _, _ = model(nodes, edges)
        an, ae, _ = sinkhorn_match(nl, el, nodes, edges, 0.12)
        correct_n += (nl.argmax(-1) == an.argmax(-1)).sum().item(); total_n += nodes.numel()
        r, c = UPPER.to(device); correct_e += (el[:, r, c].argmax(-1) == ae[:, r, c].argmax(-1)).sum().item()
        total_e += len(nodes)*len(r)
    return correct_n/total_n, correct_e/total_e

In [ ]:
EPOCHS, PATIENCE, FORCE_RETRAIN = 100, 15, False
CHECKPOINT_PATH = MODELS_DIR / 'graphvae_qm9.pt'
CONFIG = {'max_nodes': MAX_NODES, 'node_types': ATOM_NAMES, 'edge_types': BOND_NAMES,
          'latent': model.latent, 'split': (N_TRAIN, N_VAL, SEED)}
history = {'train': [], 'val': []}; best_val = float('inf'); start = time.time()
if CHECKPOINT_PATH.exists() and not FORCE_RETRAIN:
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=True)
    if checkpoint['config'] != CONFIG:
        raise ValueError('Incompatible checkpoint; set FORCE_RETRAIN=True to replace it.')
    model.load_state_dict(checkpoint['model_state']); best_val = checkpoint['best_val']
    print(f'Loaded reusable checkpoint: {CHECKPOINT_PATH}')
else:
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-6, fused=USE_AMP)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=5)
    stale = 0
    for epoch in range(1, EPOCHS+1):
        train_metrics = run_epoch(train_loader, optimizer, epoch)
        val_metrics = run_epoch(val_loader, epoch=epoch); scheduler.step(val_metrics['loss'])
        history['train'].append(train_metrics['loss']); history['val'].append(val_metrics['loss'])
        if val_metrics['loss'] < best_val:
            best_val, stale = val_metrics['loss'], 0
            torch.save({'model_state': model.state_dict(), 'config': CONFIG, 'best_val': best_val}, CHECKPOINT_PATH)
        else: stale += 1
        print(f"{epoch:03d} train={train_metrics['loss']:.4f} val={val_metrics['loss']:.4f} "
              f"node={val_metrics['node']:.3f} edge={val_metrics['edge']:.3f} kl={val_metrics['kl']:.3f}")
        if stale >= PATIENCE: print('Early stopping.'); break
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=True)
    model.load_state_dict(checkpoint['model_state'])
node_acc, edge_acc = reconstruction_accuracy(test_loader)
print(f'Test matched accuracy: nodes={node_acc:.3%}, edges={edge_acc:.3%}; elapsed={(time.time()-start)/60:.1f} min')

## 6. Sample and inspect generated graphs

We draw $z\sim\mathcal N(0,I)$, take modal node/edge classes, remove edges touching padding, and visualize graphs. The simple diagnostics below measure nonempty and connected outputs plus an approximate valence check. They are not substitutes for RDKit sanitization, uniqueness, or novelty metrics.

In [ ]:
@torch.no_grad()
def sample_graphs(n=256, temperature=0.8):
    model.eval(); z = torch.randn(n, model.latent, device=device)
    nl, el = model.decoder(z)
    nodes = torch.distributions.Categorical(logits=nl/temperature).sample()
    edges = torch.distributions.Categorical(logits=el/temperature).sample()
    edges = torch.triu(edges, diagonal=1); edges = edges + edges.transpose(1, 2)
    present = nodes.ne(0); edges = edges*present[:, :, None]*present[:, None, :]
    return nodes.cpu(), edges.cpu()

def graph_diagnostics(nodes, edges):
    bond_order = torch.tensor([0., 1., 2., 3., 1.5])
    max_valence = torch.tensor([0., 4., 3., 2., 1.])
    nonempty = connected = valence_ok = 0
    for x, a in zip(nodes, edges):
        ids = torch.where(x > 0)[0]
        if len(ids) == 0: continue
        nonempty += 1; seen = {int(ids[0])}; frontier = list(seen)
        while frontier:
            i = frontier.pop()
            for j in torch.where(a[i] > 0)[0].tolist():
                if j not in seen: seen.add(j); frontier.append(j)
        connected += len(seen) == len(ids)
        valence = bond_order[a].sum(-1)
        valence_ok += bool((valence[ids] <= max_valence[x[ids]] + 1e-6).all())
    return {'nonempty': nonempty/len(nodes), 'connected': connected/len(nodes), 'valence_proxy': valence_ok/len(nodes)}

generated_nodes, generated_edges = sample_graphs()
print(graph_diagnostics(generated_nodes, generated_edges))

In [ ]:
def draw_graph(ax, nodes, edges):
    ids = torch.where(nodes > 0)[0]; n = len(ids)
    if n == 0: ax.set_title('empty'); ax.axis('off'); return
    angles = torch.linspace(0, 2*math.pi, n+1)[:-1]
    xy = torch.stack([angles.cos(), angles.sin()], 1)
    colors = {'C':'#555555', 'N':'#4f7cff', 'O':'#e34a4a', 'F':'#55aa55'}
    for u in range(n):
        for v in range(u+1, n):
            bond = int(edges[ids[u], ids[v]])
            if bond: ax.plot([xy[u,0], xy[v,0]], [xy[u,1], xy[v,1]], color='black', lw=1+0.7*min(bond,3), zorder=1)
    for i, node_id in enumerate(ids):
        name = ATOM_NAMES[int(nodes[node_id])]
        ax.scatter(xy[i,0], xy[i,1], s=500, color=colors[name], zorder=2)
        ax.text(xy[i,0], xy[i,1], name, color='white', ha='center', va='center', weight='bold', zorder=3)
    ax.set_aspect('equal'); ax.axis('off')

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, nodes, edges in zip(axes.flat, generated_nodes[:8], generated_edges[:8]): draw_graph(ax, nodes, edges)
plt.suptitle('Samples from the GraphVAE prior'); plt.tight_layout(); plt.show()

## 7. Interpreting the reproduction

### Faithfully reproduced

- graph-convolutional variational encoder and Gaussian latent space;
- non-autoregressive probabilistic decoding of a fixed-size graph;
- joint node/edge reconstruction after approximate graph matching;
- QM9 heavy-atom graphs with typed atoms and bonds;
- prior sampling and graph-level diagnostics.

### Differences and limitations

- Sinkhorn matching replaces the paper's approximate max-pooling matching algorithm.
- Architecture, class weights, optimization, and split assignment are modern reproduction choices.
- Padding/no-edge are categorical classes here; the paper parameterizes existence and labels with separate probabilities.
- Modal or categorical decoding does not guarantee connectedness, valence, or chemical validity.
- Approximate valence is not RDKit sanitization; rigorous molecular evaluation should convert outputs to molecules and report validity, uniqueness, novelty, and property distributions.
- A fixed nine-node canvas is practical for QM9 but scales quadratically and cannot generate larger molecules.

Useful experiments include KL/capacity schedules to combat posterior collapse, hard Hungarian matching at evaluation, constrained decoding, latent interpolation, and comparison against autoregressive graph generators.

## Reference

Simonovsky M, Komodakis N. *GraphVAE: Towards Generation of Small Graphs Using Variational Autoencoders.* ICANN 2018; arXiv:1802.03480.